# SchemaQuake GRPO Training (Colab T4 ready)

End-to-end GRPO LoRA fine-tuning of a small instruct model on the SchemaQuake environment.

- Default model: **Qwen2.5-0.5B-Instruct** (fits on free T4)
- For L4: try `Qwen2.5-1.5B`. For A100: try `Qwen2.5-3B`.

**Run cells top to bottom.**

## 1. GPU + clone repo

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists('/content/schemaquake/.git'):
    !rm -rf /content/schemaquake
    !git clone https://github.com/ambujraj2001/SchemaQuake.git /content/schemaquake
else:
    !cd /content/schemaquake && git pull
%cd /content/schemaquake

## 2. Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q "trl>=0.11" "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "datasets>=2.20" "bitsandbytes>=0.43" wandb
!pip install -q "openenv-core>=0.2.3"
!pip install -q -e "/content/schemaquake[dev,agents]"

## 3. Fix Python path so `schemaquake` resolves to the real package

Colab can mistake the cloned folder `/content/schemaquake` for the package itself. We force imports to use the real package under `src/schemaquake`.

In [ ]:
import sys, os

for name in list(sys.modules):
    if name == 'schemaquake' or name.startswith('schemaquake.'):
        del sys.modules[name]

sys.path = [p for p in sys.path if p not in ('', '/content', '/content/schemaquake')]
sys.path.insert(0, '/content/schemaquake/src')

assert os.path.exists('/content/schemaquake/src/schemaquake/__init__.py'), 'package missing'

import schemaquake
print('schemaquake at:', schemaquake.__file__)

from schemaquake.env import SchemaQuakeEnv
from schemaquake.types import SQAction, ToolName
from schemaquake.prompts import SYSTEM_PROMPT
print('imports OK')

## 4. Quick sanity test (optional)

In [ ]:
%cd /content/schemaquake
!pytest -q

## 5. Load model with Unsloth

On free **T4** GPU, keep `Qwen2.5-0.5B-Instruct`. Upgrade only on bigger GPUs.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)

## 6. Episode rollout + reward extraction

We wrap one SchemaQuake episode into `(prompt, completion, scalar reward)` for GRPO.

In [ ]:
import json, re

MAX_STEPS = 10
_TOOL_JSON_RE = re.compile(r'\{[^{}]*?\}', re.DOTALL)

def parse_action(text: str) -> SQAction:
    m = _TOOL_JSON_RE.search(text)
    if not m:
        return SQAction(tool=ToolName.NOOP, confidence=0.0)
    try:
        d = json.loads(m.group(0))
        return SQAction(
            tool=ToolName(d.get('tool','noop')),
            args=d.get('args') or {},
            confidence=d.get('confidence'),
        )
    except Exception:
        return SQAction(tool=ToolName.NOOP, confidence=0.0)

def build_prompt(history):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}]
    for turn in history:
        msgs.append(turn)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def rollout_one(seed: int):
    env = SchemaQuakeEnv(max_steps=MAX_STEPS, p_no_drift=0.2)
    obs = env.reset(seed=seed, episode_id=f'tr-{seed}')
    history = [{'role':'user','content':json.dumps(obs.episode_brief)}]
    generations = []
    while not obs.done:
        prompt = build_prompt(history)
        ids = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=64, do_sample=True, temperature=0.7, top_p=0.9)
        text = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
        generations.append(text)
        act = parse_action(text)
        obs = env.step(act)
        history.append({'role':'assistant','content':text})
        history.append({'role':'user','content':json.dumps(obs.tool_result)[:1200]})
    total = (obs.reward_breakdown or {}).get('total', 0.0)
    return build_prompt(history[:1]), '\n'.join(generations), float(total), obs.reward_breakdown

_p, _g, _r, _bd = rollout_one(0)
print('sample rollout reward =', _r)
print('breakdown =', _bd)

## 7. GRPO Trainer

Tiny smoke-test config. Increase `NUM_PROMPTS` and `num_generations` if VRAM allows.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

NUM_PROMPTS = 32
train_ds = Dataset.from_list([{'prompt': f'seed:{i}', 'seed': i} for i in range(NUM_PROMPTS)])

def schemaquake_reward(completions, prompts=None, **kw):
    rewards = []
    for p in prompts:
        seed = int(p.split(':')[-1])
        _, _, total, _ = rollout_one(seed)
        rewards.append(total)
    return rewards

cfg = GRPOConfig(
    output_dir='/content/runs/schemaquake_grpo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=128,
    logging_steps=1,
    save_steps=50,
    report_to='none',
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=schemaquake_reward,
    args=cfg,
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model('/content/runs/schemaquake_grpo/final')

## 8. Save adapter to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/schemaquake_grpo
!cp -r /content/runs/schemaquake_grpo/final /content/drive/MyDrive/schemaquake_grpo/
print('saved to Drive')

## 9. Baseline eval (random vs heuristic)

In [ ]:
%cd /content/schemaquake
!python -m eval.run_eval --agent random --episodes 20
!python -m eval.run_eval --agent heuristic --episodes 20